In [24]:
from lakehouse import LakeConfig, connect

lake = LakeConfig.from_env()
con = connect()   # DuckDB already authenticated on MinIO (minio:9000)

gold_ml = lake.uri("gold",  "gld_ml_sales") + ".parquet"
gold_ml_orders = lake.uri("gold",  "gld_ml_orders") + ".parquet"

In [13]:
from lakehouse import LakeConfig, connect
lake = LakeConfig.from_env()
u = lambda layer, m: lake.uri(layer, f"{m}.parquet")
connect().sql(f"""
WITH gold AS (
    SELECT order_date, mark_cod, order_chanel_type, sum(total_orders_count) AS orders_summed
    FROM read_parquet('{u('gold', 'gld_ml_sales')}') GROUP BY ALL
),
real AS (
    SELECT cab.order_date, cab.mark_cod, cab.order_chanel_type, count(DISTINCT det.order_id) AS orders_real
    FROM read_parquet('{u('silver', 'slv_order_det')}') det
    JOIN read_parquet('{u('silver', 'slv_order_cab')}') cab USING (order_id)
    JOIN read_parquet('{u('silver', 'slv_product')}') p ON p.prod_spk = det.prod_sfk
    WHERE upper(trim(cab.order_origin)) = 'PARFOIS' AND upper(trim(det.type)) = 'SALE'
      AND p.cat_des_en NOT IN ('Informática', 'Marketing', 'Others', 'Consumable', 'Works')
    GROUP BY ALL
)
SELECT mark_cod, sum(orders_summed) AS orders_summed, sum(orders_real) AS orders_real,
       round(sum(orders_summed) / sum(orders_real), 3) AS inflation
FROM gold JOIN real USING (order_date, mark_cod, order_chanel_type)
GROUP BY ALL ORDER BY orders_real DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────┬─────────────┬───────────┐
│ mark_cod  │ orders_summed │ orders_real │ inflation │
│  varchar  │    int128     │   int128    │  double   │
├───────────┼───────────────┼─────────────┼───────────┤
│ Online ES │       2784899 │     2073748 │     1.343 │
│ Online PT │        973766 │      725530 │     1.342 │
│ Online FR │        290284 │      221050 │     1.313 │
│ Online PL │        176062 │      132442 │     1.329 │
│ Online IT │        148988 │      108669 │     1.371 │
│ Online UK │        104524 │       70880 │     1.475 │
│ Online DE │         66609 │       42164 │      1.58 │
│ Online IE │         47827 │       34320 │     1.394 │
│ Online HU │         38768 │       25435 │     1.524 │
│ Online NL │         34369 │       22761 │      1.51 │
│     ·     │            ·  │          ·  │        ·  │
│     ·     │            ·  │          ·  │        ·  │
│     ·     │            ·  │          ·  │        ·  │
│ Online LV │          4072 │        2481 │     

In [14]:
from lakehouse import LakeConfig, connect
lake = LakeConfig.from_env()
connect().sql(f"""
    SELECT mark_cod, sum(total_orders_count) AS orders
    FROM read_parquet('{lake.uri('gold', 'gld_ml_orders.parquet')}')
    WHERE mark_cod IN ('Online ES', 'Online PT')
    GROUP BY ALL ORDER BY 1
""").show()

┌───────────┬─────────┐
│ mark_cod  │ orders  │
│  varchar  │ int128  │
├───────────┼─────────┤
│ Online ES │ 2073748 │
│ Online PT │  725530 │
└───────────┴─────────┘



In [18]:
from lakehouse import LakeConfig, connect
lake = LakeConfig.from_env()
connect().sql(f"""
    SELECT count(DISTINCT order_id)
    FROM read_parquet('{lake.uri('silver', 'slv_order_cab.parquet')}')
    WHERE mark_cod IN ('Online ES')
    
""").show()

┌──────────────────────────┐
│ count(DISTINCT order_id) │
│          int64           │
├──────────────────────────┤
│                  2073748 │
└──────────────────────────┘



In [19]:
from lakehouse import LakeConfig, connect
lake = LakeConfig.from_env()
connect().sql(f"""
    SELECT count(DISTINCT order_id)
    FROM read_parquet('{lake.uri('silver', 'slv_order_cab.parquet')}')
    WHERE mark_cod IN ('Online PT')
    
""").show()

┌──────────────────────────┐
│ count(DISTINCT order_id) │
│          int64           │
├──────────────────────────┤
│                   725530 │
└──────────────────────────┘



In [22]:

df_T= connect().sql(f"""
    SELECT *
    FROM read_parquet('{gold_ml}')
    
""").to_df()

df_T

,order_date,mark_cod,order_chanel_type,cat_des_en,macro_cat_group,year,month,day_of_week,is_weekend,total_prod_qty_sold,...,daily_market_paid_media,paid_media_rolling_7d_avg,qty_sold_lag_1d,qty_sold_lag_7d,qty_sold_rolling_7d_avg,cat_share_market_rolling_30d,macro_event_type,general_campaign_type,promo_urgency_level,active_campaigns_count
0,2024-01-01,Online AT,APP,Hand Bag,BAGS_AND_SMALL_GOODS,2024,1,1,0,1,...,0.864,NaN,<NA>,<NA>,NaN,NaN,NONE,NONE,0,0
1,2024-01-01,Online AT,APP,Wallet,BAGS_AND_SMALL_GOODS,2024,1,1,0,2,...,0.864,NaN,<NA>,<NA>,NaN,NaN,NONE,NONE,0,0
2,2024-01-01,Online AT,DESKTOP,Apparel,WINTER_AND_APPAREL,2024,1,1,0,7,...,0.864,NaN,<NA>,<NA>,NaN,NaN,NONE,NONE,0,0
3,2024-01-01,Online AT,DESKTOP,Belts,BAGS_AND_SMALL_GOODS,2024,1,1,0,1,...,0.864,NaN,<NA>,<NA>,NaN,NaN,NONE,NONE,0,0
4,2024-01-01,Online AT,DESKTOP,Jewellery,JEWELLERY_AND_WATCHES,2024,1,1,0,3,...,0.864,NaN,<NA>,<NA>,NaN,NaN,NONE,NONE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508501,2025-08-05,Online IE,APP,Sunglasses,SUMMER_AND_TRAVEL,2025,8,2,0,0,...,178.104,162.172286,0,0,0.142857,0.009755,NONE,PRICE_PROMO_CLEARANCE,2,1
508502,2025-08-05,Online IE,APP,Travel,SUMMER_AND_TRAVEL,2025,8,2,0,0,...,178.104,162.172286,0,0,0.000000,0.014524,NONE,PRICE_PROMO_CLEARANCE,2,1
508503,2025-08-05,Online IE,APP,Wallet,BAGS_AND_SMALL_GOODS,2025,8,2,0,0,...,178.104,162.172286,1,0,1.857143,0.088879,NONE,PRICE_PROMO_CLEARANCE,2,1
508504,2025-08-05,Online IE,APP,Watches,JEWELLERY_AND_WATCHES,2025,8,2,0,0,...,178.104,162.172286,1,0,0.142857,0.012573,NONE,PRICE_PROMO_CLEARANCE,2,1


In [25]:

df_T= connect().sql(f"""
    SELECT *
    FROM read_parquet('{gold_ml_orders}')
    
""").to_df()

df_T

,order_date,mark_cod,order_chanel_type,year,month,day_of_week,is_weekend,total_orders_count,daily_market_paid_media,paid_media_rolling_7d_avg,orders_count_lag_1d,orders_count_lag_7d,orders_count_rolling_7d_avg,macro_event_type,general_campaign_type,promo_urgency_level,active_campaigns_count
0,2024-01-01,Online AT,APP,2024,1,1,0,1,0.864,NaN,<NA>,<NA>,NaN,NONE,NONE,0,0
1,2024-01-01,Online AT,DESKTOP,2024,1,1,0,2,0.864,NaN,<NA>,<NA>,NaN,NONE,NONE,0,0
2,2024-01-01,Online BE,APP,2024,1,1,0,2,12.594,NaN,<NA>,<NA>,NaN,NONE,NONE,0,0
3,2024-01-01,Online BE,DESKTOP,2024,1,1,0,5,12.594,NaN,<NA>,<NA>,NaN,NONE,NONE,0,0
4,2024-01-01,Online DE,APP,2024,1,1,0,11,31.992,NaN,<NA>,<NA>,NaN,NONE,NONE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30324,2025-12-31,Online SI,DESKTOP,2025,12,3,0,6,0.414,0.950571,12,3,7.714286,NONE,PRICE_PROMO_CLEARANCE,2,1
30325,2025-12-31,Online SK,APP,2025,12,3,0,0,2.814,2.989714,0,0,0.142857,NONE,PRICE_PROMO_CLEARANCE,2,1
30326,2025-12-31,Online SK,DESKTOP,2025,12,3,0,1,2.814,2.989714,1,0,1.857143,NONE,PRICE_PROMO_CLEARANCE,2,1
30327,2025-12-31,Online UK,APP,2025,12,3,0,20,265.506,288.905143,40,3,45.571429,NONE,PRICE_PROMO_CLEARANCE,2,1


In [26]:
from lakehouse import LakeConfig, connect
lake = LakeConfig.from_env()
for m in ["gld_ml_sales", "gld_ml_orders"]:
    n = connect().sql(f"""
        SELECT count(*) FROM (
            SELECT order_date < lag(order_date) OVER (ORDER BY file_row_number) AS out_of_order
            FROM read_parquet('{lake.uri('gold', m + '.parquet')}', file_row_number = true)
        ) WHERE out_of_order
    """).fetchone()[0]
    print(m, "rows out of order:", n)

gld_ml_sales rows out of order: 0
gld_ml_orders rows out of order: 0
